In [28]:
pip install -q requests beautifulsoup4 pandas tqdm

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

c:\Users\mrgan\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
API_URL = "https://understandingwar.org/wp-json/facetwp/v1/refresh"
PER_PAGE = 10            
MAX_RETRIES = 5          
SLEEP_BETWEEN = 0.5      


def build_payload(page: int) -> dict:
    """Build the JSON payload for a given FacetWP page number."""
    return {
        "action": "facetwp_refresh",
        "data": {
            "facets": {
                "product_line_search": "",
                "date_from": [],
                "publication_types_filter": ["update"],
                "map_type": [],
                "map_series": [],
                "clear": [],
                "pagination": [],
            },
            "frozen_facets": {},
            "http_params": {
                "get": [],
                "uri": "analysis/russia-ukraine/russian-offensive-campaign-assessment",
                "url_vars": {
                    "publication_types_filter": ["update"],
                    "search_date_order": "post_date_desc",
                },
            },
            "template": "product_line_research",
            "extras": {
                "sort": "default",
            },
            "soft_refresh": 1,
            "is_bfcache": 1,
            "first_load": 0,
            "paged": page,
        },
    }


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0.0.0 Safari/537.36"
    ),
    "Content-Type": "application/json",
    "Accept": "application/json, text/plain, */*",
    "Referer": (
        "https://understandingwar.org/analysis/russia-ukraine/"
        "russian-offensive-campaign-assessment/"
        "?_publication_types_filter=update"
    ),
    "Origin": "https://understandingwar.org",
}

In [ ]:
def parse_cards(html: str) -> list:
    """
    Parse the FacetWP template HTML and return a list of
    {'date': ..., 'link': ...} dicts.

    The API returns cards like:
        <p class="research-card-post-date">Jul 23, 2026</p>
        <h3 class="... research-card-title"><a href="...">Title</a></h3>
    """
    soup = BeautifulSoup(html, "html.parser")
    records = []

    for card in soup.select(".research-card-loop-item-3colgrid"):

        date_el = card.select_one(".research-card-post-date")
        date_text = date_el.get_text(strip=True) if date_el else ""

        link_el = card.select_one(
            "h3.research-card-title a, "
            "h3.gb-headline a, "
            "a.research-card-btn, "
            "a.txt-link"
        )
        link = link_el["href"] if link_el and link_el.has_attr("href") else ""
        if link.startswith("/"):
            link = "https://understandingwar.org" + link

        if date_text or link:
            records.append({"date": date_text, "link": link})

    return records


def fetch_page(page: int, session: requests.Session) -> dict:
    """Fetch one page from the FacetWP API with exponential-backoff retries."""
    payload = build_payload(page)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.post(API_URL, json=payload, headers=HEADERS, timeout=30)
            resp.raise_for_status()
            return resp.json()
        except (requests.RequestException, ValueError) as exc:
            wait = 2 ** attempt
            print(f"  Page {page}, attempt {attempt}/{MAX_RETRIES} failed: {exc}")
            if attempt < MAX_RETRIES:
                print(f"    Retrying in {wait}s …")
                time.sleep(wait)
            else:
                raise

In [ ]:
session = requests.Session()

print("Fetching page 1 to determine total pages …")
first = fetch_page(1, session)

pager = first.get("settings", {}).get("pager", {})
total_pages = int(pager.get("total_pages", 1))
total_rows = int(pager.get("total_rows", 0))
print(f"Total updates : {total_rows:,}")
print(f"Total pages   : {total_pages}")
print(f"Per page      : {PER_PAGE}")

Fetching page 1 to determine total pages …
Total updates : 10,580
Total pages   : 353
Per page      : 10


In [33]:
all_records = []

template_html = first.get("template", "")
page1_records = parse_cards(template_html)
all_records.extend(page1_records)
print(f"Page 1: {len(page1_records)} records")

# Pages 2 … N
for page_num in tqdm(range(2, total_pages + 1), desc="Scraping pages"):
    data = fetch_page(page_num, session)
    html = data.get("template", "")
    records = parse_cards(html)
    all_records.extend(records)
    time.sleep(SLEEP_BETWEEN)

print(f"\n✅ Total records scraped: {len(all_records):,}")

Page 1: 30 records


Scraping pages:  15%|█▌        | 54/352 [01:05<05:43,  1.15s/it]

  ⚠ Page 56, attempt 1/5 failed: 429 Client Error: Too Many Requests for url: https://understandingwar.org/wp-json/facetwp/v1/refresh
    Retrying in 2s …


Scraping pages:  16%|█▋        | 58/352 [01:12<06:31,  1.33s/it]

  ⚠ Page 60, attempt 1/5 failed: 429 Client Error: Too Many Requests for url: https://understandingwar.org/wp-json/facetwp/v1/refresh
    Retrying in 2s …


...

  ⚠ Page 344, attempt 1/5 failed: 429 Client Error: Too Many Requests for url: https://understandingwar.org/wp-json/facetwp/v1/refresh
    Retrying in 2s …


Scraping pages:  99%|█████████▉| 350/352 [08:39<00:02,  1.31s/it]

  ⚠ Page 352, attempt 1/5 failed: 429 Client Error: Too Many Requests for url: https://understandingwar.org/wp-json/facetwp/v1/refresh
    Retrying in 2s …


Scraping pages: 100%|██████████| 352/352 [08:44<00:00,  1.49s/it]


✅ Total records scraped: 10,580


In [34]:
df = pd.DataFrame(all_records, columns=["date", "link"])

df["date"] = pd.to_datetime(df["date"], format="%b %d, %Y", errors="coerce")

df.sort_values("date", ascending=False, inplace=True, ignore_index=True)
df.drop_duplicates(subset=["date", "link"], inplace=True, ignore_index=True)

print(f"Dataset shape: {df.shape}")
df.head(20)

Dataset shape: (10580, 2)


,date,link
0,2026-07-23,https://understandingwar.org/research/russia-u...
1,2026-07-22,https://understandingwar.org/research/russia-u...
2,2026-07-21,https://understandingwar.org/research/russia-u...
3,2026-07-20,https://understandingwar.org/research/russia-u...
4,2026-07-19,https://understandingwar.org/research/russia-u...
5,2026-07-18,https://understandingwar.org/research/russia-u...
6,2026-07-17,https://understandingwar.org/research/russia-u...
7,2026-07-16,https://understandingwar.org/research/russia-u...
8,2026-07-15,https://understandingwar.org/research/russia-u...
9,2026-07-14,https://understandingwar.org/research/russia-u...


In [35]:
OUTPUT_FILE = "isw_russian_offensive_updates.csv"
df.to_csv(OUTPUT_FILE, index=False)
